In [1]:
from content.connection.credentials import read_credentials
from content.connection.connect_db import create_conection
from content.extraction.read_db import read_file
from content.db.querys import *
import os
import shutil
import pandas as pd
import re
from datetime import datetime
from psycopg2.extras import execute_batch
import psycopg2
from itertools import cycle
import time

In [2]:
credentials = read_credentials()
engine_local = create_conection(credentials)

Conexion exitosa a la DB


# EXTRACTION
## Lectura de archivos 

* Se realiza la lectura de las dos carpetas, el repositorio que contiene el historico de archivos recibidos y la carpeta donde se esta dejando todo lo nuevo


In [3]:
path_new = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Nueva_asignacion'
path_old = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Repositorio'

#df_mail_repository, df_sinfin_repository = read_file(path_old)
df_mail_new, df_sinfin_new = read_file(path_new)

Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Nueva_asignacion


# TRANSFORM

In [4]:
#print(len(df_sinfin_repository))
print(len(df_sinfin_new))

477


Se ordena los df para eliminar registros duplicados, dejando solo la asignación mas resiente, se agrega en el df resultado el email que viene originalmente en otra pestaña

In [5]:
def join_df_mail_sinfin(df_mail, df_sifin):
    df_mail = df_mail.sort_values(by=['CTA CONTR', 'FECHA DE CESION'], ascending=[True, False])
    df_mail = df_mail.drop_duplicates(subset=['CTA CONTR'])
    df_sifin = df_sifin.sort_values(by=['NUMERO_CUENTA', 'DATE1'], ascending=[True, False])
    df_sifin = df_sifin.drop_duplicates(subset=['NUMERO_CUENTA'])
    df_mail = df_mail[['CTA CONTR', 'CORREO ELECTRONICO']]
    df_repository = pd.merge(df_mail, df_sifin, how='right', left_on='CTA CONTR', right_on='NUMERO_CUENTA')

    df_repository['EmailsDeudor'] = df_repository['CORREO ELECTRONICO']

    df_repository = df_repository.drop(columns=['CTA CONTR', 'CORREO ELECTRONICO'])

    return df_repository

In [6]:
#df_consolidated_old = join_df_mail_sinfin(df_mail_repository, df_sinfin_repository)
df_consolidated_new = join_df_mail_sinfin(df_mail_new, df_sinfin_new)

In [7]:
print(len(df_consolidated_new))

477


TABLA RESUMEN DE CAMPÁÑAS Y MÁS

In [17]:
df_group_read=df_consolidated_new.groupby(['name_file', 'TEXT6']).agg(
    Q = ('PRIMER_NOMBRE', 'count')
)

#df_group_read['Q'] = df_group_read['Q'].astype(str)
df_group_read = df_group_read.sort_values(by=['name_file', 'Q'], ascending=[True, False])
print(df_group_read)
total = df_group_read['Q'].sum()
print(f'Total obligaciones recibidas:               {total}')

                                              Q
name_file       TEXT6                          
11_12_2024.xlsx 01372450XXFINTEL1001006776  211
                013724FINDOMNUE3006770       77
                242100048530                 53
                244100048542                 38
                243100048532                 32
                01372450XXFINTEL6006006778   27
                013724FINDOMTOD1406765       22
                013724FINDOMTOD6006783       17
Total obligaciones recibidas:               477


In [9]:
#df_consolidated_old.to_sql('asignacion', engine_local, if_exists='append', index=False, schema='sinfin')

In [143]:
text_query = read_asignation_naturgy()

df_asignation_db = pd.read_sql_query(text_query, engine_local)
print(len(df_asignation_db))


167108


In [142]:
df_row_asignated = pd.merge(df_asignation_db, df_consolidated_new, on='NUMERO_CUENTA', how='inner')

ValueError: You are trying to merge on object and int64 columns for key 'NUMERO_CUENTA'. If you wish to proceed you should use pd.concat

In [20]:
columns_diference = df_row_asignated[['NUMERO_CUENTA', 'name_file_x','TEXT8_x', 'TEXT6_x', 'MONEY1_x', 'DATE1_x', 'DATE2_x', 'TEXT8_y', 'TEXT6_y','MONEY1_y', 'DATE1_y', 'DATE2_y', 'name_file_y']]
columns_diference['DIAS'] = (columns_diference['DATE1_y'] - columns_diference['DATE2_x']).dt.days

columns_diference['MES'] = columns_diference['DATE2_x'].dt.month
columns_diference = columns_diference[columns_diference['DIAS'] < 0]
columns_diference = columns_diference[columns_diference['MES'] >= 8]

columns_diference['NUMERO_CUENTA'] = columns_diference['NUMERO_CUENTA'].astype(str)
def validate_file(row):
    
    if row['name_file_x'] == row['name_file_y']:
        return 'Si'
    else:
        return 'No'

columns_diference['Val']=columns_diference.apply(validate_file, axis=1)
columns_diference = columns_diference[columns_diference['Val'] == 'No']

C:\Users\jherrera\AppData\Local\Temp\ipykernel_27508\638582966.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  columns_diference['DIAS'] = (columns_diference['DATE1_y'] - columns_diference['DATE2_x']).dt.days
C:\Users\jherrera\AppData\Local\Temp\ipykernel_27508\638582966.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  columns_diference['MES'] = columns_diference['DATE2_x'].dt.month


In [21]:
print(len(df_row_asignated))

218


ES LA CANTIDAD DE FILAS QUE LEYÓ DE LA CARPETA NUEVA_ASIGNACION

In [22]:
df_data_update = pd.merge(df_consolidated_new, df_asignation_db, on='NUMERO_CUENTA')

len(df_data_update)

218

ES LA CANTIDAD DE LOS NUEVOS CASOS 

In [23]:
df_new_data = pd.merge(df_consolidated_new, df_asignation_db, on='NUMERO_CUENTA', how='left', indicator=True)
df_new_data = df_new_data[df_new_data['_merge']== 'left_only']
df_new_data = df_new_data[df_new_data['_merge'] == 'left_only'].drop(columns=['_merge'])
print(len(df_new_data))

259


In [24]:
def clear_df(df):
    columns_drop = [col for col in df.columns if col.endswith('_y')]
    df = df.drop(columns=columns_drop)
    df.columns = [col.replace('_x','') for col in df.columns]
    df = df.reset_index()
    df['TEXT10'] = pd.NA
    
    df = df.dropna(subset=['NUMERO_CUENTA'])
    try:
        df = df.drop(columns=['Dirección'])
    except:
        print('')
        
    try:
          df = df.drop(columns=['index'])  
    except:
        print('')
        
    print(list(df.columns))
    print(len(df))
    
    return df

REALIZA LA ASIGNACION POR SUCURSAL RESPONSABLE

In [ ]:
df_new_data = clear_df(df_new_data)
df_data_update = clear_df(df_data_update)
df_data_update['sucursal_responsable'] = 'Asignar'



['IDENTIFICACION', 'TIPO_DOC', 'PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'SEXO', 'ESTADO_CIVIL', 'PERSONAS_A_CARGO', 'FECHA_NAC', 'IDIOMA', 'EMPRESA', 'CARGO', 'PROFESION', 'TIPO_VIVIENDA', 'NUMERO_CUENTA', 'PRODUCTO_ID', 'CLIENTE_ID', 'CUENTA_CLIENTE_ID', 'TEXT1', 'TEXT2', 'TEXT3', 'TEXT4', 'TEXT5', 'TEXT6', 'TEXT7', 'TEXT8', 'TEXT9', 'TEXT10', 'TEXT11', 'TEXT12', 'TEXT13', 'TEXT14', 'TEXT15', 'TEXT16', 'TEXT17', 'TEXT18', 'TEXT19', 'TEXT20', 'MONEY1', 'MONEY2', 'MONEY3', 'MONEY4', 'MONEY5', 'MONEY6', 'MONEY7', 'MONEY8', 'MONEY9', 'MONEY10', 'MONEY11', 'MONEY12', 'MONEY13', 'MONEY14', 'MONEY15', 'MONEY16', 'MONEY17', 'MONEY18', 'MONEY19', 'MONEY20', 'NUMBER1', 'NUMBER2', 'NUMBER3', 'NUMBER4', 'NUMBER5', 'PERCENT1', 'PERCENT2', 'PERCENT3', 'DATE1', 'DATE2', 'DATE3', 'DATE4', 'DATE5', 'DATE6', 'DATE7', 'Dir1Deudor', 'Ciudad1Deudor', 'Dpto1Deudor', 'Barrio1Deudor', 'TelesDeudor', 'Dir2Deudor', 'Ciudad2Deudor', 'Dpto2Deudor', 'Barrio2Deudor', 'EmailsDeudor

In [31]:
df_new_data['NUMERO_CUENTA'] = pd.to_numeric(df_new_data['NUMERO_CUENTA'], errors='coerce')
df_new_data['ENTIDAD_ID'] = 'NATURGY'

# load
## Actualizar filas con la información de la nueva asignación

In [29]:
con = psycopg2.connect("dbname=Estrategia user=CDM password=password host=localhost port=5432")
cur = con.cursor()
data = df_data_update[[
    'MONEY1',
    'MONEY4',
    'DATE1',
    'DATE2',
    'TEXT1',
    'TEXT2',
    'TEXT3',
    'TEXT4',
    'TEXT5',
    'TEXT6',
    'TEXT7',
    'TEXT8',
    'TEXT9',
    'name_file',
    'EmailsDeudor',
    'sucursal_responsable',
    'NUMERO_CUENTA'
    

]].values.tolist()
# Ejecutar las actualizaciones en bloque
execute_batch(cur, """
    UPDATE sinfin.asignacion
    SET "MONEY1" = %s,
        "MONEY4" = %s,
        "DATE1" = %s,
        "DATE2" = %s,
        "TEXT1" = %s,
        "TEXT2" = %s,
        "TEXT3" = %s,
        "TEXT4" = %s,
        "TEXT5" = %s,
        "TEXT6" = %s,
        "TEXT7" = %s,
        "TEXT8" = %s,
        "TEXT9" = %s,
        "name_file" = %s,
        "EmailsDeudor" = %s,
        "sucursal_responsable" = %s
    WHERE "NUMERO_CUENTA" = %s
""", data)

con.commit()
cur.close()
con.close()

# Cargar nuevas filas de la asignación

In [32]:
df_new_data.to_sql('asignacion', engine_local, if_exists='append', index=False, schema='sinfin')

259

REALIZA EL MOVIEMIENTO DE LA CARPETA NUEVA ASIGNACION A REPOSITORIO

In [33]:
#path_new
list_file_new_asignation = os.listdir(path_new)
for name in list_file_new_asignation:
    name_file_source = os.path.join(path_new, name)
    name_file_destination = os.path.join(path_old, name)
    shutil.move(name_file_source, name_file_destination )
    
print(f'Se realizo el movimiento de {len(list_file_new_asignation)} archivo a la carpeta del repositorio.')

Se realizo el movimiento de 1 archivo a la carpeta del repositorio.


# FUNCIONES

In [34]:
def transform_df(df):
    date_today = datetime.today()
    df['DATE1'] = pd.to_datetime(df['DATE1'], errors='coerce')
    df['TEXT6'] = df['TEXT6'].apply(lambda x: x.strip() if isinstance(x, str) else x)
    df['MONEY1'] = df['MONEY1'].round(2)
    df['MONEY2'] = df['MONEY2'].round(2)
    df['MONEY3'] = df['MONEY3'].round(2)
    df['MONEY4'] = df['MONEY4'].round(2)
    

    def change_typedata(df):
        column_convert = ['NUMERO_CUENTA', 'TEXT2', 'TEXT4', 'TEXT6', 'TEXT9', 'Barrio1Deudor', 'PRODUCTO_ID']
        for col in column_convert:
            if col in df.columns:
                df[col] = df[col].astype(str).apply(lambda x: re.sub(r'\.0$', '', x))
            else:
                raise KeyError(f"La columna {col} no existe en el DataFrame")
        
        
        df['DATE1'] = pd.to_datetime(df['DATE1'], errors='coerce')
        df['DATE2'] = pd.to_datetime(df['DATE2'], errors='coerce')
        
        df['PRODUCTO_ID'] = df['TEXT6']

        
        return df
    
    def filter_estatus_date(df):
        df['TEXT20'] = df['DATE2'].apply(lambda x: 'Retirado' if x < date_today else 'Vigente')
        df = df[df['TEXT20'] == 'Vigente']
        
        return df


    df = change_typedata(df)
    df = filter_estatus_date(df)

    new_order_columns = ['IDENTIFICACION', 'TIPO_DOC', 'PRIMER_NOMBRE', 'SEGUNDO_NOMBRE', 'PRIMER_APELLIDO', 'SEGUNDO_APELLIDO', 'SEXO', 'ESTADO_CIVIL', 'PERSONAS_A_CARGO', 'FECHA_NAC', 'IDIOMA', 'EMPRESA', 'CARGO', 'PROFESION', 'TIPO_VIVIENDA', 'NUMERO_CUENTA', 'PRODUCTO_ID', 'CLIENTE_ID', 'CUENTA_CLIENTE_ID', 'TEXT1', 'TEXT2', 'TEXT3', 'TEXT4', 'TEXT5', 'TEXT6', 'TEXT7', 'TEXT8', 'TEXT9', 'TEXT10', 'TEXT11', 'TEXT12', 'TEXT13', 'TEXT14', 'TEXT15', 'TEXT16', 'TEXT17', 'TEXT18', 'TEXT19', 'TEXT20', 'MONEY1', 'MONEY2', 'MONEY3', 'MONEY4', 'MONEY5', 'MONEY6', 'MONEY7', 'MONEY8', 'MONEY9', 'MONEY10', 'MONEY11', 'MONEY12', 'MONEY13', 'MONEY14', 'MONEY15', 'MONEY16', 'MONEY17', 'MONEY18', 'MONEY19', 'MONEY20', 'NUMBER1', 'NUMBER2', 'NUMBER3', 'NUMBER4', 'NUMBER5', 'PERCENT1', 'PERCENT2', 'PERCENT3', 'DATE1', 'DATE2', 'DATE3', 'DATE4', 'DATE5', 'DATE6', 'DATE7', 'Dir1Deudor', 'Ciudad1Deudor', 'Dpto1Deudor', 'Barrio1Deudor', 'TelesDeudor', 'Dir2Deudor', 'Ciudad2Deudor', 'Dpto2Deudor', 'Barrio2Deudor', 'EmailsDeudor', 'DirEmpDeudor', 'CiudadEmpDeudor', 'DptoEmpDeudor', 'TelesEmpDeudor', 'IdentConyuge', 'Nombrecy', 'Dir1cy', 'Ciudad1cy', 'Dpto1cy', 'Telescy', 'Emailscy', 'IdentCodeudor1', 'NombreCodeudor1', 'Dir1Codeudor1', 'Ciudad1Codeudor1', 'Dpto1Codeudor1', 'TelesCodeudor1', 'EmailsCodeudor1', 'IdentRef1', 'NombreRef1', 'Dir1Ref1', 'Ciudad1Ref1', 'Dpto1Ref1', 'TelesRef1', 'EmailsRef1', 'IdentRef2', 'NombreRef2', 'Dir1Ref2', 'Ciudad1Ref2', 'Dpto1Ref2', 'TelesRef2', 'EmailsRef2','sucursal_responsable', 'ejecutivo_responsable', 'date_entered']
    df = df.reindex(columns = new_order_columns)
    
    
    return df

In [35]:
def update_data_asignation():
    text_query = read_asignation_naturgy()

    df_data_total_asignation = pd.read_sql_query(text_query, engine_local)
    print(f'Total filas asignación: {len(df_data_total_asignation)}')

    return df_data_total_asignation

In [36]:
def estado_pago(row):
    if row['fecha_pago'] >= row['DATE1'] <= row['DATE2']:
        return 'Si'
    else:
        return 'No'

In [37]:
def aplicated_payment(row):
    
    if row['payment'] > 0 and row['estado_pago'] == 'Si':
        reclamado = row['payment']
        pendiente = round(row['MONEY1'] - row['payment'],2)
        return pd.Series([reclamado, pendiente], index=['reclamado', 'pendiente'])
    else:
        return pd.Series([0, row['MONEY4']], index=['reclamado', 'pendiente'])

In [38]:
text_query = read_dictionary_campain()

In [39]:
def update_tipo(df):
    text_query = read_dictionary_campain()
    df_dictionary_campain = pd.read_sql_query(text_query, engine_local)
    df_merge = pd.merge(df, df_dictionary_campain, left_on= "TEXT8", right_on= "Vuelta", how='left', indicator=True)
    new_name_campain = df_merge[df_merge['_merge'] == 'left_only']
    new_name_campain = new_name_campain.drop_duplicates(subset='TEXT8')
    new_name_campain = new_name_campain['TEXT8']
    new_name_campain.to_excel('nuevas_campañas_errado.xlsx', index=False)
    #df_merge= df_merge.drop(['date_entered', 'name_file', 'Vuelta_correcta', '_merge'], axis=1)
    
    df_merge.to_excel('df_merge.xlsx', index=False)
    
    return df_merge

In [40]:
def update_data_db(df):
    
    
    text_dic = """
        SELECT      *
        FROM        sinfin.dic_naturgy
    """
    df_dic_naturgy = pd.read_sql_query(text_dic, engine_local)

    text_payment = """
        SELECT      date_entered as fecha_pago,  
                    "ACCOUNT_NUMBER",
                    "PAYMENT_AMOUNT"
        FROM        data_sinfin.pagos
        WHERE       "ENTIDAD_ID" = 'NATURGY'
    """
    df_payments = pd.read_sql_query(text_payment, engine_local)
    
    df_total_saldo = pd.merge (df, df_dic_naturgy, left_on='TEXT8', right_on= 'Vuelta', how='left')
    
    df_total_saldo['TEXT8'] = df_total_saldo['Vuelta_correcta']
    df_total_saldo['TEXT2'] = df_total_saldo['Tipo']
    df_total_saldo['TEXT2'] = df_total_saldo['Tipo'].apply(lambda x: 'BAJA - VENCIDA' if pd.isnull(x) else x)
    

    df_payment_sum = df_payments.groupby(['ACCOUNT_NUMBER', 'fecha_pago']).agg(
        payment = ('PAYMENT_AMOUNT', 'sum')
        ).reset_index()
    print(f'cantidad de filas en pagos {len(df_payment_sum)}')
    

    df_asignation_payments_aplicated = pd.merge(df_total_saldo, df_payment_sum, left_on='NUMERO_CUENTA', right_on='ACCOUNT_NUMBER', indicator=True, how='left')

    df_asignation_payments_aplicated['estado_pago'] = df_asignation_payments_aplicated.apply(estado_pago, axis=1)

    df_asignation_payments_aplicated[['MONEY2', 'MONEY4']] = df_asignation_payments_aplicated.apply(aplicated_payment, axis=1)
    
    df_asignation_payments_aplicated=df_asignation_payments_aplicated.drop(columns=['payment', '_merge'])
    
    print(f'Tamaño del df transformado: {len(df_asignation_payments_aplicated)}')

    
    return df_asignation_payments_aplicated

In [41]:
def save_file_predictive(df, name):
    
    path_file_predictive = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo'
    name_file = os.path.join(path_file_predictive, f'{name}.xlsx')
    df.to_excel(name_file, sheet_name='Predictivo', index= False)
    print(name_file)

In [42]:
def save_file(df, name):

    df['TEXT10'] = df['sucursal_responsable']
    
    print(len(df))
    df = df.sort_values(by='fecha_pago', ascending=False)
    df=df.drop(columns=['sucursal_responsable', 'ejecutivo_responsable', 'date_entered', 'index', 'Vuelta', 'Tipo', 'Vuelta_correcta', 'ACCOUNT_NUMBER', 'fecha_pago', 'estado_pago'])
    df=df.drop_duplicates(subset=['NUMERO_CUENTA'])
    print(len(df))
    
    
    path_sinfin = r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Sinfin'
    name_file_sinfin = os.path.join(path_sinfin,f'{name}.xlsx')
    print(name_file_sinfin)
    df.to_excel(name_file_sinfin, index=False)

In [80]:
def preasignated(df):
    import numpy as np
    
    
    df_q_nomina = pd.read_sql_query(query_nomina(), engine_local)
    print(df_q_nomina)
    q_gestor = df_q_nomina['q_gestores'].sum()
   
    def calculate_participation(row, q_gestor):
        return row / q_gestor
        
    df_q_nomina['participacion'] = df_q_nomina['q_gestores'] .apply(lambda x: calculate_participation(x, q_gestor))
    print(df_q_nomina)
    
    n = len(df)
    option = df_q_nomina['sucursal'].unique().tolist()
    porcentajes = df_q_nomina['participacion'].unique().tolist()
    
    df['ciudad'] = np.random.choice(option, size= n, p=porcentajes)
    return(df)

In [83]:
df_preasignated = df_sinfin_new[df_sinfin_new['MONEY1'] > 500]
df_preasignated=preasignated(df_preasignated)
df_preasignated = df_preasignated[['IDENTIFICACION', 'NUMERO_CUENTA', 'TEXT6', 'TEXT1', 'TEXT2', 'MONEY1','DATE2', 'TelesDeudor', 'ciudad']]

df_preasignated_bogota = df_preasignated[df_preasignated['ciudad'] == 'BOGOTA']
df_preasignated_cali = df_preasignated[df_preasignated['ciudad'] == 'CALI']
df_preasignated_madrid = df_preasignated[df_preasignated['ciudad'] == 'MADRID']
df_resume = df_preasignated.groupby('ciudad').agg(
    Q = ('ciudad', 'count'),
    Suma = ('MONEY1', 'sum'),
    Ticket = ('MONEY1', 'mean')
    ).reset_index()
print(F'\nTotal filas a preasignar: {len(df_preasignated)},\nDistribución por oficina:\n{df_resume}\n')

df_preasignated_cali.to_excel(r'Z:\3 Equipo Cali\Preasignada\df_preasignated_cali.xlsx', index=False)
df_preasignated_madrid.to_excel(r'Z:\3 Equipo Cali\Preasignada\df_preasignated_madrid.xlsx', index=False)
df_preasignated_bogota.to_excel(r'Z:\2. Equipo Bogotá\Preasignada\df_preasignated_bogota.xlsx', index=False)

  sucursal  q_gestores
0   BOGOTA          10
1   MADRID           3
2     CALI           7
  sucursal  q_gestores  participacion
0   BOGOTA          10           0.50
1   MADRID           3           0.15
2     CALI           7           0.35

Total filas a preasignar: 33,
Distribución por oficina:
   ciudad   Q      Suma       Ticket
0  BOGOTA  17  16903.76   994.338824
1    CALI  11  14148.21  1286.200909
2  MADRID   5   3981.40   796.280000



C:\Users\jherrera\AppData\Local\Temp\ipykernel_27508\4247143673.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ciudad'] = np.random.choice(option, size= n, p=porcentajes)


In [86]:
def asignated_office(df):
    
    
    def print_time_excute(text, init, end):
        time_execute = end - init
        print(f'Se ha finalizado la ejecución de {text}, en {time_execute}')
        
    def split_teles(df):
        time_init=time.time()
        df_teles = df[['NUMERO_CUENTA', 'TelesDeudor']]
        df_teles_separado = df_teles['TelesDeudor'].str.split('|', expand=True)
        df_teles = pd.concat([df_teles[['NUMERO_CUENTA']], df_teles_separado], axis=1)
        
        df_teles.columns=['NUMERO_CUENTA', 'Tel1', 'Tel2', 'Tel3', 'Tel4', 'Tel5']
        df_predictive = pd.merge(df, df_teles, on='NUMERO_CUENTA')
        
        list_columns = ['NUMERO_CUENTA',
                        'IDENTIFICACION',
                        'PRIMER_NOMBRE',
                        'TEXT1',
                        'TEXT2',
                        'TEXT6',
                        'TEXT8',
                        'MONEY1',
                        'MONEY4',
                        'DATE1',
                        'DATE2',
                        'TelesDeudor',
                        'sucursal_responsable',
                        'Dir1Deudor',
                        'Ciudad1Deudor',
                        'EmailsDeudor',
                        'Tel1',
                        'Tel2',
                        'Tel3',
                        'Tel4',
                        'Tel5',
                        'ejecutivo_responsable',
                        'date_entered'
                        ]
        df_predictive=df_predictive[list_columns]
        
        time_end = time.time()
        print_time_excute('Separación telefonos', time_init, time_end)
        
        return df_predictive
    
    def sucursal_responsable(df):
        df.to_excel('df.xlsx', index=False)
        time_init=time.time()
        
        df = df.sort_values(by=['TEXT2', 'TEXT6', 'DATE2', 'MONEY1'], ascending=True)
        
        dfs={}
        df_activa = df[df['TEXT2'] == 'ACTIVA - VIGENTE']
        dfs['activa'] = df_activa 
        df_baja = df[df['TEXT2'] == 'BAJA - VENCIDA']
        dfs['baja'] = df_baja
        list_sucursal = ['Cali', 'Bogota']
        
        print(f'dic dfs: {len(dfs)}')
        print(f'Tamaño df activa: {len(dfs['activa'])}')
        
        for index, df in dfs.items():
            max_row = len(df)
            print(f'filas a asignar: {max_row} en: {index}')
            list_sucursal_cycle = cycle(list_sucursal)
            df['sucursal_responsable'] =   [next(list_sucursal_cycle) for _ in range(max_row)]
            
        df = pd.concat(dfs)
       
        time_end = time.time()
        print_time_excute('Asignación sucursal', time_init, time_end)
        
        return df
        
    def procesing_data_upload(df_updata):
        time_init=time.time()
        
        list_data = df_updata[[
        'sucursal_responsable',
        'date_entered',
        'ejecutivo_responsable',
        'NUMERO_CUENTA'
        ]].values.tolist()

        text_update = """
                UPDATE sinfin.asignacion
                SET "sucursal_responsable" = %s,
                    "date_entered" = %s,
                    "ejecutivo_responsable" = %s
                WHERE "NUMERO_CUENTA" = %s
                """
        
        time_end = time.time()
        
        update_data_sql(list_data, text_update)
        print_time_excute('Actualización en BD', time_init, time_end)
            
    def update_data_sql(df_data, text_update):
        time_init=time.time()
        
        con = psycopg2.connect("dbname=Estrategia user=CDM password=password host=localhost port=5432")
        cur = con.cursor()
        data = df_data
        # Ejecutar las actualizaciones en bloque
        execute_batch(cur, text_update, data)

        con.commit()
        cur.close()
        con.close()
    
    
    df_asignated = df[(df['sucursal_responsable'].isna()) | (df['sucursal_responsable']=='Asignar') ]
    
    row_asignated = len(df_asignated)
    if row_asignated > 0:
        print(f'Se realizara la asignación de: {row_asignated}')
        df = sucursal_responsable(df_asignated)
        
        df = update_data_db(df)
        
        save_file(df, 'Sinfin')
        
        procesing_data_upload(df)
    else:
        print(f'No se realiza la asignación de sucursales, se actualizara el saldo')
        
        df = update_tipo(df)
        df = update_data_asignation()
        df = transform_df(df)
        df = update_data_db(df)
        save_file(df, 'Sinfin')
        df = split_teles(df)
    
        
    list_city_Castilla = [
        'alaquàs',
        'albal',
        'albalat de la ribera',
        'alborache, alcàsser',
        'lalcúdia',
        'aldaia',
        'alfafar',
        'alfarb',
        'algemesí',
        'alginet',
        'almussafes',
        'benetússer',
        'benifaió',
        'beniparrell',
        'benicull de xúquer',
        'bétera',
        'bugarra',
        'buñol',
        'camporrobles',
        'carlet',
        'catadau',
        'catarroja',
        'caudete de las fuentes',
        'corbera',
        'quart de poblet',
        'cullera',
        'chera',
        'cheste',
        'xirivella',
        'chiva',
        'favara',
        'fortaleny',
        'fuenterrobles',
        'godelleta',
        'guadassuar',
        'llíria',
        'loriguilla'

    ]
    
    list_city_Valencia = [
        'valencia',
        'llaurí',
        'llombai',
        'macastre',
        'manises',
        'massanassa',
        'mislata',
        'montserrat',
        'montroi',
        'paiporta',
        'paterna',
        'pedralba',
        'picanya',
        'picassent',
        'polinyà de xúquer',
        'real',
        'requena',
        'riba-roja de túria',
        'riola',
        'sedaví',
        'siete aguas',
        'silla',
        'sinarcas',
        'sollana',
        'sot de chera',
        'sueca',
        'tavernes de la valldigna',
        'torrent',
        'turís',
        'utiel',
        'yátova',
        'benicull de xúquer',
        'llocnou de la corona'
    ]
    
       
    df['Ciudad1Deudor'] = df['Ciudad1Deudor'].str.strip().str.lower()
    list_city_Valencia = [city.lower().strip() for city in list_city_Valencia]
    
     
    df['filter'] = df['Ciudad1Deudor'].isin(list_city_Castilla)
    df['filter'] = df['Ciudad1Deudor'].isin(list_city_Valencia)
        
    df.to_excel('df_filter.xlsx', index=False)
    print('filtro guardado')
        
    df = df[df['filter'] == False]
    print(f'tamaño filtrado {len(df)}')
    df.drop('filter', inplace=True, axis=1)   
     
    df = df.drop_duplicates(subset='NUMERO_CUENTA')
    df = df.sort_values(by=['DATE2', 'MONEY1'], ascending=[True, False])
    df = df[df['MONEY4'] > 0]
    
    
    #Filtrar exclusiones 
    df_exclusiones = pd.read_excel(r'Z:\1. Coordinadores\Exclusiones\Exclusiones Consolidado.xlsx')
    df = pd.merge(df, df_exclusiones, left_on= 'NUMERO_CUENTA', right_on='Caso', indicator=True, how='left')
    df = df[df['_merge'] == 'left_only']
    df.drop('_merge', inplace=True, axis=1)
    
    print(df['NUMERO_CUENTA'].dtypes)
    #Filtrar casos de seguros ue no se deben cobrar
    df_seguros = pd.read_excel(r'Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Seguros 132 no cobrar.xlsx')
    print(df_seguros['Cuenta contrato'].dtypes)
    df_seguros['Cuenta contrato']=df_seguros['Cuenta contrato'].astype(str)
    df = pd.merge(df, df_seguros, left_on= 'NUMERO_CUENTA', right_on='Cuenta contrato', indicator=True, how='left')
    
    df = df[df['_merge'] == 'left_only']
    df.drop(['_merge',
             'Cuenta contrato',	
             'Nombre',	
             'Reclamado',	
             'Campaña',	
             'Otros',	
             'pendiente',
             'Caso',
             'Cartera',	
             'Vuelta',	
             'importe',	
             'DNI',	
             'Telefono 1',	
             'Telefono 2'#,	
             #'Telefono'
             ], inplace=True, axis=1)
    df_baja = df[df['TEXT2'] == 'BAJA - VENCIDA']
    df_baja_ = df_baja[df_baja['MONEY1'] >= 100]
    df_baja_menores = df_baja[df_baja['MONEY1'] < 100]
    
    df_activa = df[df['TEXT2'] == 'ACTIVA - VIGENTE']
    df_activa_ = df_activa[df_activa['MONEY1'] >= 100]
    df_activa_menores = df_activa[df_activa['MONEY1'] < 100]
    
    save_file_predictive(df_baja_, 'df_baja')
    save_file_predictive(df_baja_menores, 'df_baja_menores')
    save_file_predictive(df_activa_, 'df_activa')
    save_file_predictive(df_activa_menores, 'df_activa_menores')
    
    
"""                    
    df_baja_bogota = df[(df['sucursal_responsable'] == 'Bogota') & (df['TEXT2'] == 'BAJA - VENCIDA')]
    df_baja_cali = df[(df['sucursal_responsable'] == 'Cali') & (df['TEXT2'] == 'BAJA - VENCIDA')]
    df_activa_bogota = df[(df['sucursal_responsable'] == 'Bogota') & (df['TEXT2'] == 'ACTIVA - VIGENTE')]
    df_activa_cali = df[(df['sucursal_responsable'] == 'Cali') & (df['TEXT2'] == 'ACTIVA - VIGENTE')]
    
    dic_df = {
        'baja_bogota' : df_baja_bogota,
        'baja_cali' : df_baja_cali,
        'activa_bogota' : df_activa_bogota,
        'activa_cali' : df_activa_cali
        }
    
    for key, value in dic_df.items():
        name_file = key
        save_file_predictive(value, name_file)
"""
        


"                    \n    df_baja_bogota = df[(df['sucursal_responsable'] == 'Bogota') & (df['TEXT2'] == 'BAJA - VENCIDA')]\n    df_baja_cali = df[(df['sucursal_responsable'] == 'Cali') & (df['TEXT2'] == 'BAJA - VENCIDA')]\n    df_activa_bogota = df[(df['sucursal_responsable'] == 'Bogota') & (df['TEXT2'] == 'ACTIVA - VIGENTE')]\n    df_activa_cali = df[(df['sucursal_responsable'] == 'Cali') & (df['TEXT2'] == 'ACTIVA - VIGENTE')]\n    \n    dic_df = {\n        'baja_bogota' : df_baja_bogota,\n        'baja_cali' : df_baja_cali,\n        'activa_bogota' : df_activa_bogota,\n        'activa_cali' : df_activa_cali\n        }\n    \n    for key, value in dic_df.items():\n        name_file = key\n        save_file_predictive(value, name_file)\n"

## Terminar actualización

In [87]:
df_data_total_asignation = update_data_asignation()

df_data_total_asignation = transform_df(df_data_total_asignation)

asignated_office(df_data_total_asignation)


Total filas asignación: 167108
No se realiza la asignación de sucursales, se actualizara el saldo
Total filas asignación: 167108
cantidad de filas en pagos 18953
Tamaño del df transformado: 32423
32423
31533
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Sinfin\Sinfin.xlsx
Se ha finalizado la ejecución de Separación telefonos, en 0.21639394760131836
filtro guardado
tamaño filtrado 34972
object
int64
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\df_baja.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\df_baja_menores.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\df_activa.xlsx
Z:\1. Coordinadores\Asignaciones\Naturgy\Bases\Predictivo\df_activa_menores.xlsx


# lectura y carga de PN

In [100]:
from pathlib import Path
path_asignation = r'Z:\1. Coordinadores\Asignaciones\PN Italia\Sinfin'
list_file_asignation = os.listdir(path_asignation)
files = []

for file in list_file_asignation:
    path_file = os.path.join(path_asignation, file)
    file_asignation = pd.read_excel(path_file)
    files.append(file_asignation)
    
df_asignation_pn = pd.concat(files)

print(len(df_asignation_pn))

25720


In [ ]:
df_asignation_db['NUMERO_CUENTA'] = df_asignation_db['NUMERO_CUENTA'].astype(str)
df_new_pn = pd.merge(df_asignation_db, df_asignation_pn, left_on='NUMERO_CUENTA', right_on="NUMERO_CUENTA", how='right', indicator=True)

df_new_pn = df_new_pn[df_new_pn['_merge'] == 'right_only']


#df_new_pn['NUMERO_CUENTA'] = df_new_pn["NUMERO_CUENTA_y"].astype(str)

list_columns ={
    'IDENTIFICACION_y' : 'IDENTIFICACION',
    'TIPO_DOC_y' : 'TIPO_DOC',
    'PRIMER_NOMBRE_y' : 'PRIMER_NOMBRE',
    'SEGUNDO_NOMBRE_y' : 'SEGUNDO_NOMBRE',
    'PRIMER_APELLIDO_y' : 'PRIMER_APELLIDO',
    'SEGUNDO_APELLIDO_x' : 'SEGUNDO_APELLIDO',
    'SEXO_y' : 'SEXO',
    'ESTADO_CIVIL_y' : 'ESTADO_CIVIL',
    'PERSONAS_A_CARGO_y' : 'PERSONAS_A_CARGO',
    'FECHA_NAC_y' : 'FECHA_NAC',
    'IDIOMA_y' : 'IDIOMA',
    'EMPRESA_y' : 'EMPRESA',
    'CARGO_y' : 'CARGO',
    'PROFESION_y' : 'PROFESION',
    'TIPO_VIVIENDA_y' : 'TIPO_VIVIENDA',
    #'NUMERO_CUENTA_y' : 'NUMERO_CUENTA',
    'PRODUCTO_ID_y' : 'PRODUCTO_ID',
    'CLIENTE_ID_y' : 'CLIENTE_ID',
    'CUENTA_CLIENTE_ID_y' : 'CUENTA_CLIENTE_ID',
    'TEXT1_y' : 'TEXT1',
    'TEXT2_y' : 'TEXT2',
    'TEXT3_y' : 'TEXT3',
    'TEXT4_y' : 'TEXT4',
    'TEXT5_y' : 'TEXT5',
    'TEXT6_y' : 'TEXT6',
    'TEXT7_y' : 'TEXT7',
    'TEXT8_y' : 'TEXT8',
    'TEXT9_y' : 'TEXT9',
    'TEXT10_y' : 'TEXT10',
    'TEXT11_y' : 'TEXT11',
    'TEXT12_y' : 'TEXT12',
    'TEXT13_y' : 'TEXT13',
    'TEXT14_y' : 'TEXT14',
    'TEXT15_y' : 'TEXT15',
    'TEXT16_y' : 'TEXT16',
    'TEXT17_y' : 'TEXT17',
    'TEXT18_y' : 'TEXT18',
    'TEXT19_y' : 'TEXT19',
    'TEXT20_y' : 'TEXT20',
    'MONEY1_y' : 'MONEY1',
    'MONEY2_y' : 'MONEY2',
    'MONEY3_y' : 'MONEY3',
    'MONEY4_y' : 'MONEY4',
    'MONEY5_y' : 'MONEY5',
    'MONEY6_y' : 'MONEY6',
    'MONEY7_y' : 'MONEY7',
    'MONEY8_y' : 'MONEY8',
    'MONEY9_y' : 'MONEY9',
    'MONEY10_y' : 'MONEY10',
    'MONEY11_y' : 'MONEY11',
    'MONEY12_y' : 'MONEY12',
    'MONEY13_y' : 'MONEY13',
    'MONEY14_y' : 'MONEY14',
    'MONEY15_y' : 'MONEY15',
    'MONEY16_y' : 'MONEY16',
    'MONEY17_y' : 'MONEY17',
    'MONEY18_y' : 'MONEY18',
    'MONEY19_y' : 'MONEY19',
    'MONEY20_y' : 'MONEY20',
    'NUMBER1_y' : 'NUMBER1',
    'NUMBER2_y' : 'NUMBER2',
    'NUMBER3_y' : 'NUMBER3',
    'NUMBER4_y' : 'NUMBER4',
    'NUMBER5_y' : 'NUMBER5',
    'PERCENT1_y' : 'PERCENT1',
    'PERCENT2_y' : 'PERCENT2',
    'PERCENT3_y' : 'PERCENT3',
    'DATE1_y' : 'DATE1',
    'DATE2_y' : 'DATE2',
    'DATE3_y' : 'DATE3',
    'DATE4_y' : 'DATE4',
    'DATE5_y' : 'DATE5',
    'DATE6_y' : 'DATE6',
    'DATE7_y' : 'DATE7',
    'Dir1Deudor_y' : 'Dir1Deudor',
    'Ciudad1Deudor_y' : 'Ciudad1Deudor',
    'Dpto1Deudor_y' : 'Dpto1Deudor',
    'Barrio1Deudor_y' : 'Barrio1Deudor',
    'TelesDeudor_y' : 'TelesDeudor',
    'Dir2Deudor_y' : 'Dir2Deudor',
    'Ciudad2Deudor_y' : 'Ciudad2Deudor',
    'Dpto2Deudor_y' : 'Dpto2Deudor',
    'Barrio2Deudor_y' : 'Barrio2Deudor',
    'EmailsDeudor_y' : 'EmailsDeudor',
    'DirEmpDeudor_y' : 'DirEmpDeudor',
    'CiudadEmpDeudor_y' : 'CiudadEmpDeudor',
    'DptoEmpDeudor_y' : 'DptoEmpDeudor',
    'TelesEmpDeudor_y' : 'TelesEmpDeudor',
    'IdentConyuge_y' : 'IdentConyuge',
    'Nombrecy_y' : 'Nombrecy',
    'Dir1cy_y' : 'Dir1cy',
    'Ciudad1cy_y' : 'Ciudad1cy',
    'Dpto1cy_y' : 'Dpto1cy',
    'Telescy_y' : 'Telescy',
    'Emailscy_y' : 'Emailscy',
    'IdentCodeudor1_y' : 'IdentCodeudor1',
    'NombreCodeudor1_y' : 'NombreCodeudor1',
    'Dir1Codeudor1_y' : 'Dir1Codeudor1',
    'Ciudad1Codeudor1_y' : 'Ciudad1Codeudor1',
    'Dpto1Codeudor1_y' : 'Dpto1Codeudor1',
    'TelesCodeudor1_y' : 'TelesCodeudor1',
    'EmailsCodeudor1_y' : 'EmailsCodeudor1',
    'IdentRef1_y' : 'IdentRef1',
    'NombreRef1_y' : 'NombreRef1',
    'Dir1Ref1_y' : 'Dir1Ref1',
    'Ciudad1Ref1_y' : 'Ciudad1Ref1',
    'Dpto1Ref1_y' : 'Dpto1Ref1',
    'TelesRef1_y' : 'TelesRef1',
    'EmailsRef1_y' : 'EmailsRef1',
    'IdentRef2_y' : 'IdentRef2',
    'NombreRef2_y' : 'NombreRef2',
    'Dir1Ref2_y' : 'Dir1Ref2',
    'Ciudad1Ref2_y' : 'Ciudad1Ref2',
    'Dpto1Ref2_y' : 'Dpto1Ref2',
    'TelesRef2_y' : 'TelesRef2',
    'EmailsRef2_y' : 'EmailsRef2'

}
df_new_pn = df_new_pn[list_columns.keys()]
df_new_pn = df_new_pn.rename(columns=list_columns)

df_new_pn = pd.DataFrame(df_new_pn)

df_new_pn['NUMERO_CUENTA'] = df_new_pn['TEXT3'].astype(str)
df_new_pn['TEXT5'] = df_new_pn['TEXT5'].astype(str)
df_new_pn['TEXT10'] = df_new_pn['TEXT10'].astype(str)
df_new_pn['TEXT7'] = df_new_pn['DATE3']
df_new_pn['DATE3'] = pd.NA
df_new_pn['TEXT8'] = df_new_pn['DATE4']
df_new_pn['DATE4'] = pd.NA
df_new_pn['ENTIDAD_ID'] = 'PNItalia'
df_new_pn.to_excel('df_pn.xlsx', index=False)

In [165]:
df_new_pn.to_sql('asignacion', engine_local, if_exists='append', index=False, schema='sinfin')

52

In [ ]:
text_query_gestion = """
    select		*

from		data_sinfin.gestion
where		"ENTIDAD_ID" = 'PNItalia' and date("HISTORY_DATE") BETWEEN '2024/10/01' AND '2024/10/30'
"""

df = pd.read_sql_query(text_query_gestion, engine_local)

df= df[['GESTOR_ID', 'HISTORY_DATE', 'HISTORY_DATE_END', 'ID_EFECTO']]
df['Semana'] = df['HISTORY_DATE'].dt.isocalendar().week
df['dia'] = df['HISTORY_DATE'].dt.day
df['hora_inicio'] = df['HISTORY_DATE'].dt.strftime('%H:%m')
df['hora_fin'] = df['HISTORY_DATE_END'].dt.strftime('%H:%m')
df['tiempo_gestion'] = df['hora_fin'] - df['hora_inicio']

print(df)

KeyError: 'tiempo_gestion'